# 01 — Data Exploration

**SmartEnergy Predictor — PJK-GM096**

Notebook ini melakukan eksplorasi awal terhadap dataset konsumsi listrik rumah tangga sintetis (`data/raw/household_consumption.csv`).

Tujuan utama:

1. Memahami distribusi fitur numerik dan kategorikal.
2. Mengidentifikasi missing values, outlier, dan anomali.
3. Mengeksplorasi pola temporal (harian, mingguan, musiman) dari konsumsi.
4. Mengukur korelasi antara fitur dan target `konsumsi_kwh`.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.figsize'] = (10, 5)
pd.set_option('display.max_columns', 50)

DATA_PATH = Path('..') / 'data' / 'raw' / 'household_consumption.csv'
df = pd.read_csv(DATA_PATH, parse_dates=['tanggal'])
df.head()

## 1. Shape & Tipe Data

In [ ]:
print(f'Shape: {df.shape}')
print(f'Rentang tanggal: {df.tanggal.min().date()}  ->  {df.tanggal.max().date()}')
df.info()

In [ ]:
df.describe().T

## 2. Missing Values

In [ ]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(3)
pd.DataFrame({'missing': missing, 'pct': missing_pct}).sort_values('missing', ascending=False)

## 3. Distribusi Target `konsumsi_kwh`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df['konsumsi_kwh'], kde=True, bins=60, ax=axes[0], color='#10b981')
axes[0].set_title('Distribusi konsumsi_kwh')
axes[0].set_xlabel('kWh per hari')
sns.boxplot(x=df['konsumsi_kwh'], ax=axes[1], color='#10b981')
axes[1].set_title('Boxplot konsumsi_kwh (outlier visible)')
plt.tight_layout()
plt.show()

## 4. Pola Temporal

In [ ]:
df_t = df.copy()
df_t['day_of_week'] = df_t['tanggal'].dt.day_name()
df_t['month'] = df_t['tanggal'].dt.month_name()
df_t['is_weekend'] = (df_t['tanggal'].dt.weekday >= 5).astype(int)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# (a) Daily trend (rolling 30-day mean)
daily = df_t.groupby('tanggal')['konsumsi_kwh'].mean().rolling(30, min_periods=1).mean()
axes[0, 0].plot(daily.index, daily.values, color='#10b981')
axes[0, 0].set_title('Tren harian (rolling mean 30 hari)')
axes[0, 0].set_ylabel('kWh')
axes[0, 0].tick_params(axis='x', rotation=30)

# (b) By day of week
order_dow = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
sns.boxplot(data=df_t, x='day_of_week', y='konsumsi_kwh', order=order_dow, ax=axes[0, 1])
axes[0, 1].set_title('Konsumsi per hari (Mon-Sun)')
axes[0, 1].tick_params(axis='x', rotation=30)

# (c) By month
month_order = ['January', 'February', 'March', 'April', 'May', 'June',
               'July', 'August', 'September', 'October', 'November', 'December']
month_means = df_t.groupby('month')['konsumsi_kwh'].mean().reindex(month_order)
month_means.plot(kind='bar', ax=axes[1, 0], color='#10b981')
axes[1, 0].set_title('Rata-rata konsumsi per bulan')
axes[1, 0].set_ylabel('kWh')
axes[1, 0].tick_params(axis='x', rotation=30)

# (d) Weekend vs weekday
sns.violinplot(data=df_t, x='is_weekend', y='konsumsi_kwh', ax=axes[1, 1])
axes[1, 1].set_xticklabels(['Weekday', 'Weekend'])
axes[1, 1].set_title('Weekday vs Weekend')

plt.tight_layout()
plt.show()

## 5. Hubungan Fitur vs Target

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

sns.scatterplot(data=df, x='suhu_celsius', y='konsumsi_kwh', alpha=0.25, ax=axes[0, 0])
axes[0, 0].set_title('Suhu vs Konsumsi')

sns.boxplot(data=df, x='jumlah_penghuni', y='konsumsi_kwh', ax=axes[0, 1])
axes[0, 1].set_title('Jumlah penghuni vs Konsumsi')

sns.scatterplot(data=df, x='jam_penggunaan_rata_rata', y='konsumsi_kwh', alpha=0.25, ax=axes[1, 0])
axes[1, 0].set_title('Jam penggunaan vs Konsumsi')

sns.boxplot(data=df, x='hari_libur', y='konsumsi_kwh', ax=axes[1, 1])
axes[1, 1].set_xticklabels(['Hari kerja', 'Hari libur'])
axes[1, 1].set_title('Hari libur vs Konsumsi')

plt.tight_layout()
plt.show()

## 6. Korelasi Numerik

In [ ]:
numeric_cols = ['suhu_celsius', 'hari_libur', 'jumlah_penghuni',
                'jumlah_perangkat_aktif', 'jam_penggunaan_rata_rata', 'jam', 'konsumsi_kwh']
corr = df[numeric_cols].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0, square=True)
plt.title('Heatmap Korelasi Fitur')
plt.tight_layout()
plt.show()

## 7. Identifikasi Outlier (IQR Method)

In [ ]:
def iqr_outliers(s: pd.Series, k: float = 1.5) -> pd.Series:
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - k * iqr, q3 + k * iqr
    return (s < lower) | (s > upper)

out = iqr_outliers(df['konsumsi_kwh'])
print(f'Total outlier konsumsi_kwh (IQR 1.5x): {out.sum()} ({out.mean() * 100:.2f}% dari data)')
df.loc[out].sort_values('konsumsi_kwh', ascending=False).head(10)

## 8. Insight Awal

- Distribusi konsumsi miring ke kanan (right-skewed) dengan beberapa outlier ekstrem yang harus diolah pada tahap preprocessing.
- Pola mingguan jelas: konsumsi weekend lebih tinggi daripada weekday.
- Hari libur menambah konsumsi sekitar 1.5-2 kWh.
- Suhu menunjukkan hubungan non-linear terhadap konsumsi (efek kuat di atas 30 C — dugaan AC).
- Jumlah penghuni dan jam penggunaan adalah dua fitur paling korelatif terhadap target.

Insight ini akan dijadikan dasar untuk feature engineering di notebook berikutnya.